In [1]:
# The framework prints a couple of "experimental" notices on import; quiet them.
import warnings
warnings.filterwarnings("ignore", message=r".*experimental.*")

In [2]:
import subprocess 
from pathlib import Path 

from dotenv import load_dotenv 
from agent_framework import Agent, MCPStdioTool 
from agent_framework.openai import OpenAIChatClient 

load_dotenv(override=True)

True

## Step 1: Create Agent

In [3]:
MODEL = "gpt-5.4-mini"

client = OpenAIChatClient(model=MODEL)

agent = Agent(
    client=client,
    instructions="You are a concise, friendly assistant. Reply in a single short sentence." 
)

## Step 2: Run It

In [4]:
result = await agent.run("Say hello in Tagalog.")
print(result.text)

Kumusta!


# Project: SQLite Todo Board

In [5]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.',
  'status': 'pending',
  'result': ''}]

In [6]:
board.show_board()

Goal #1: Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.

## Step 3: Add Tools

In [7]:
def show_todos() -> list[dict]:
    """ List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """ Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short lost of step descriptions. """
    return {
        'goal_id': goal_id,
        'step_ids': [board.add_step(goal_id, step) for step in steps]
    }

def complete_task(task_id: int, result: str) -> dict:
    """ Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {
        'task_id': task_id,
        'status': 'done'
    }

In [8]:
board_agent = Agent(
    client=client,
    instructions="You help manage a shared todo board.",
    tools=[show_todos,
           complete_task]
)

In [9]:
result = await board_agent.run("What is on the board right now, and what is its status?")
print(result.text)

There is 1 item on the board, and its status is pending:

- **Goal (ID 1):** Read `notes.txt`, translate its contents into Tagalog, and write the Tagalog to `tagalog.txt.`

No steps are listed yet.


## Step 4: Add MCP

In [13]:
workspace = Path("workspace").resolve()

class FilesystemMCP(MCPStdioTool):
    """ The filesystem server with its stderr sent to DEVNULL and its working
    directory set to the workspace, so file names resolve there and it runs
    cleanly from a Jupyter kernel on Windows. """

    def get_mcp_client(self):
        from mcp.client.stdio import StdioServerParameters, stdio_client
        params = StdioServerParameters(command=self.command,
                                       args=self.args,
                                       env=self.env,
                                       cwd=str(workspace))
        return stdio_client(server=params,
                            errlog=subprocess.DEVNULL)
filesystem = FilesystemMCP(
    name="filesystem",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem",
          str(workspace)]
)

In [14]:
file_agent = Agent(
    client=client,
    instructions="You can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem]
)

In [15]:
async with filesystem:
    result = await file_agent.run("Read notes.txt and summarize it in one short sentence.")
print(result.text)

The notes say the team is building a small language tutor piece by piece, with each helper carefully completing one task from a shared board.


## Step 5: Put It In A Loop With A Goal

In [16]:
INSTRUCTIONS = """ 
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. nBegin by laying out a short plan:
the handful of concrete steps the work itself breaks down into, added to the
board under the goal. Then carry them out with your file tools, marking each
step done as you finish it. Once the steps are all done, close the goal. Your
files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    client=client,
    instructions=INSTRUCTIONS,
    tools=[
        show_todos,
        plan_steps,
        complete_task,
        filesystem
    ]
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.")
board.claim_todo(goal_id)

async with filesystem:
    await worker.run("Please work the pending goal on the board.")
board.show_board()

Goal #1: Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.
  Step #2: Read notes.txt to understand the source text.  Read notes.txt and captured the source text for translation.
  Step #3: Translate the notes into Tagalog.  Translated the notes into Tagalog and saved the translation content.
  Step #4: Write the Tagalog translation to tagalog.txt.  Verified that tagalog.txt contains the translated text.
  Step #5: Verify the output and mark the goal complete.  Completed the translation goal and verified the output file.